
# Preference Extractor Layer 2 — XLM-R Base
## FRESH KAGGLE SESSION — FINAL ONE-SHOT NOTEBOOK

Use **only** the companion dataset:

`preference_layer2_dataset_KAGGLE_FRESH_R3.zip`

Dataset ZIP SHA256:

`4a37c5bc61ddfad2bcf2785c5c63cb3c93b40e745edac3f4530d3a5db2b26ad5`

This notebook is intentionally strict. It will not train unless it finds the
four uniquely named R3 JSONL files and their SHA256 fingerprints match exactly.

Protocol:
1. TRAIN = fitting only.
2. VALIDATION = checkpoint selection + confidence calibration only.
3. TEST = once after checkpoint + thresholds are frozen.
4. FINAL_HOLDOUT = not parsed until you explicitly set `RUN_FINAL_HOLDOUT=True`
   after the frozen TEST run.


In [ ]:

# 0. CONFIG
from pathlib import Path
import os, json, math, random, re, hashlib, shutil, zipfile, time
from collections import Counter, defaultdict

SEED = 20260824
EXPECTED_DATASET_VERSION = "2026-08-24-r3-kaggle-fresh"
EXPECTED_DATASET_ZIP_SHA256 = "4a37c5bc61ddfad2bcf2785c5c63cb3c93b40e745edac3f4530d3a5db2b26ad5"

EXPECTED_FILES = {
    "train": {
        "filename": "preference_layer2_r3_train.jsonl",
        "sha256": "cd8bab6da02891ae76af40f268ca674626a6a931a9d8d9e4d0fd26d7c8aa840b",
        "rows": 48000,
    },
    "validation": {
        "filename": "preference_layer2_r3_validation.jsonl",
        "sha256": "81b70cd8510fe3ba9657979283344aa37cbcfebc68d16bde15ead60f44d5da0a",
        "rows": 6000,
    },
    "test": {
        "filename": "preference_layer2_r3_test.jsonl",
        "sha256": "bfe49319df7ca531a7db5df3045697e3e564c2561bec00996a39c5326be67142",
        "rows": 6000,
    },
    "final_holdout": {
        "filename": "preference_layer2_r3_final_holdout.jsonl",
        "sha256": "9a5834803decb0d4d234a2fec2f9d905563a96765f713c88fd55834b2918ce5a",
        "rows": 4000,
    },
}

DIMENSIONS = ("cost", "power", "performance", "reliability")
LEVELS = ("VERY_LOW", "LOW", "MEDIUM", "HIGH", "VERY_HIGH")
LEVEL_TO_ID = {name: i for i, name in enumerate(LEVELS)}
BASE_MODEL_HF = "FacebookAI/xlm-roberta-base"

MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
BATCH_SIZE_DUAL_GPU = 16
GRADIENT_ACCUMULATION = 2
ENCODER_LR = 2e-5
HEAD_LR = 8e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
MONOTONICITY_WEIGHT = 0.10
MAX_GRAD_NORM = 1.0

TARGET_DIRECT_PRESENCE_ACCURACY = 0.985
TARGET_DIRECT_INTENSITY_ACCURACY = 0.985

FORCE_RETRAIN = False
RUN_FINAL_HOLDOUT = False

WORK_DIR = Path("/kaggle/working/layer2_xlmr_fresh_r3")
WORK_DIR.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = WORK_DIR / "best_model_fp32.pt"
TRAINING_HISTORY_FILE = WORK_DIR / "training_history.json"
FREEZE_FILE = WORK_DIR / "freeze_manifest.json"
ARTIFACT_DIR = WORK_DIR / "artifact"
ARTIFACT_ZIP = Path("/kaggle/working/preference_layer2_xlmr_v1_FINAL.zip")

print("Config: PASS")


In [ ]:

# 1. ENVIRONMENT PREFLIGHT
import numpy as np
import pandas as pd
import torch
import sklearn
import transformers
from safetensors.torch import save_file as save_safetensors

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("sklearn:", sklearn.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Select Kaggle GPU T4 x2.")

GPU_COUNT = torch.cuda.device_count()
print("GPU count:", GPU_COUNT)
for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / 1024**3, 1), "GB")

if GPU_COUNT < 2:
    raise RuntimeError("Select Kaggle accelerator GPU T4 x2 before training.")

DEVICE = torch.device("cuda:0")
BATCH_SIZE = BATCH_SIZE_DUAL_GPU

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print("Environment preflight: PASS")


In [ ]:

# 2. FIND EXACT R3 DATASET ROOT BY UNIQUE FILENAMES + HASHES
KAGGLE_INPUT = Path("/kaggle/input")

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def count_jsonl_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

def verify_candidate_root(root: Path):
    data_dir = root / "data"
    if not data_dir.exists():
        return False, "no data/ directory"

    for split, spec in EXPECTED_FILES.items():
        p = data_dir / spec["filename"]
        if not p.exists():
            return False, f"missing {spec['filename']}"

        actual_sha = sha256_file(p)
        if actual_sha != spec["sha256"]:
            return False, (
                f"{spec['filename']} SHA mismatch: "
                f"{actual_sha} != {spec['sha256']}"
            )

        actual_rows = count_jsonl_rows(p)
        if actual_rows != spec["rows"]:
            return False, (
                f"{spec['filename']} row mismatch: "
                f"{actual_rows} != {spec['rows']}"
            )

    return True, "all four split hashes + row counts match"

# Kaggle may expose the uploaded dataset already extracted.
train_name = EXPECTED_FILES["train"]["filename"]
train_candidates = list(KAGGLE_INPUT.rglob(train_name))

print("Unique R3 train candidates:", len(train_candidates))
for p in train_candidates:
    print(" -", p)

candidate_roots = []
for train_path in train_candidates:
    # expected layout root/data/train.jsonl
    root = train_path.parent.parent
    ok, reason = verify_candidate_root(root)
    print("Candidate root:", root)
    print("  verify:", ok, reason)
    if ok:
        candidate_roots.append(root)

# Also support the rare case where the raw ZIP itself is present.
if not candidate_roots:
    zip_name = "preference_layer2_dataset_KAGGLE_FRESH_R3.zip"
    zip_candidates = list(KAGGLE_INPUT.rglob(zip_name))

    for zp in zip_candidates:
        actual_zip_sha = sha256_file(zp)
        print("ZIP candidate:", zp)
        print("  SHA256:", actual_zip_sha)

        if actual_zip_sha != EXPECTED_DATASET_ZIP_SHA256:
            continue

        extract_root = Path("/kaggle/working/r3_dataset_extract")
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True)

        with zipfile.ZipFile(zp, "r") as z:
            z.extractall(extract_root)

        ok, reason = verify_candidate_root(extract_root)
        print("  extracted verify:", ok, reason)
        if ok:
            candidate_roots.append(extract_root)

if len(candidate_roots) != 1:
    raise RuntimeError(
        "Could not identify exactly one valid R3 dataset root.\n"
        f"Valid roots found: {candidate_roots}\n"
        "In a NEW Kaggle session, attach only "
        "preference_layer2_dataset_KAGGLE_FRESH_R3.zip."
    )

DATASET_ROOT = candidate_roots[0]
DATA_DIR = DATASET_ROOT / "data"
FILES = {
    split: DATA_DIR / spec["filename"]
    for split, spec in EXPECTED_FILES.items()
}

print("SELECTED DATASET ROOT:", DATASET_ROOT)
print("Exact four-file fingerprint verification: PASS")


In [ ]:

# 3. VERIFY METADATA + QA
metadata_path = DATASET_ROOT / "dataset_metadata.json"
qa_path = DATASET_ROOT / "QA_REPORT.json"

if not metadata_path.exists() or not qa_path.exists():
    raise RuntimeError("R3 metadata/QA files missing. DO NOT TRAIN.")

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
qa_report = json.loads(qa_path.read_text(encoding="utf-8"))

if metadata.get("version") != EXPECTED_DATASET_VERSION:
    raise RuntimeError(
        f"Wrong R3 metadata version: {metadata.get('version')}"
    )

if not qa_report.get("all_checks_pass", False):
    raise RuntimeError("R3 QA report is not PASS. DO NOT TRAIN.")

required_checks = [
    "schema_and_evidence_pass",
    "within_split_exact_text_unique",
    "cross_split_exact_text_overlap_zero",
    "cross_split_structural_overlap_zero",
    "cross_split_sample_id_overlap_zero",
    "cross_split_template_id_overlap_zero",
    "runaway_length_absent",
]

for key in required_checks:
    if not qa_report["critical_checks"].get(key, False):
        raise RuntimeError(f"R3 critical QA failed: {key}")

print("Metadata version:", metadata["version"])
print("Length safety:", metadata["length_safety"])
print("QA all_checks_pass: PASS")


In [ ]:

# 4. LOAD TRAIN / VALIDATION / TEST ONLY
def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise RuntimeError(f"Bad JSON: {path.name}:{line_no}") from exc
    return rows

train_rows = read_jsonl(FILES["train"])
validation_rows = read_jsonl(FILES["validation"])
test_rows = read_jsonl(FILES["test"])

ROWS = {
    "train": train_rows,
    "validation": validation_rows,
    "test": test_rows,
}

for split, rows in ROWS.items():
    if len(rows) != EXPECTED_FILES[split]["rows"]:
        raise RuntimeError(f"{split}: row count mismatch")

print({k: len(v) for k, v in ROWS.items()})
print("FINAL_HOLDOUT has NOT been parsed.")


In [ ]:

# 5. INDEPENDENT SCHEMA / EVIDENCE / LEAKAGE QA
def normalize_exact(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().casefold())

def normalize_structural(text: str) -> str:
    text = text.casefold()
    text = re.sub(r"\d+(?:[.,]\d+)?", "<num>", text)
    return re.sub(r"\s+", " ", text.strip())

for split, rows in ROWS.items():
    seen = set()
    for row in rows:
        key = normalize_exact(row["text"])
        if key in seen:
            raise RuntimeError(f"{split}: duplicate exact text")
        seen.add(key)

        if set(row["labels"]) != set(DIMENSIONS):
            raise RuntimeError(f"{row['sample_id']}: bad dimensions")

        calculated_l1 = int(any(row["labels"][d]["presence"] == 1 for d in DIMENSIONS))
        if row["layer1_expected"] != calculated_l1:
            raise RuntimeError(f"{row['sample_id']}: bad layer1_expected")

        for d in DIMENSIONS:
            lab = row["labels"][d]
            if lab["presence"] == 0:
                if not (
                    lab["intensity"] is None
                    and lab["intensity_label"] is None
                    and lab["intensity_mask"] == 0
                ):
                    raise RuntimeError(f"{row['sample_id']}/{d}: NO_SIGNAL invariant")

            if lab["intensity_mask"] == 1:
                if lab["presence"] != 1:
                    raise RuntimeError(f"{row['sample_id']}/{d}: intensity without presence")
                if lab["intensity_label"] not in LEVEL_TO_ID:
                    raise RuntimeError(f"{row['sample_id']}/{d}: bad level")
                if lab["intensity"] != LEVEL_TO_ID[lab["intensity_label"]]:
                    raise RuntimeError(f"{row['sample_id']}/{d}: bad ordinal id")

            ev = lab.get("evidence")
            if ev is not None and ev not in row["text"]:
                raise RuntimeError(f"{row['sample_id']}/{d}: evidence mismatch")

        for rel in row["relations"]:
            if rel["higher"] == rel["lower"]:
                raise RuntimeError(f"{row['sample_id']}: self relation")
            if rel["evidence"] not in row["text"]:
                raise RuntimeError(f"{row['sample_id']}: relation evidence mismatch")

exact_sets = {
    split: {normalize_exact(r["text"]) for r in rows}
    for split, rows in ROWS.items()
}
struct_sets = {
    split: {normalize_structural(r["text"]) for r in rows}
    for split, rows in ROWS.items()
}
id_sets = {
    split: {r["sample_id"] for r in rows}
    for split, rows in ROWS.items()
}
template_sets = {
    split: {r["template_id"] for r in rows}
    for split, rows in ROWS.items()
}

names = list(ROWS)
for i, a in enumerate(names):
    for b in names[i+1:]:
        checks = {
            "exact": len(exact_sets[a] & exact_sets[b]),
            "structural": len(struct_sets[a] & struct_sets[b]),
            "id": len(id_sets[a] & id_sets[b]),
            "template": len(template_sets[a] & template_sets[b]),
        }
        if any(checks.values()):
            raise RuntimeError(f"Leakage {a}<->{b}: {checks}")

print("Independent schema/evidence QA: PASS")
print("TRAIN/VALIDATION/TEST exact + structural leakage: 0")


In [ ]:

# 6. DISTRIBUTION + RAW LENGTH AUDIT
distribution = []

for split, rows in ROWS.items():
    langs = Counter(r["language"] for r in rows)
    print(split, "languages:", dict(langs))

    word_lengths = np.array([len(r["text"].split()) for r in rows])
    print(
        split,
        "word lengths:",
        {
            "p50": int(np.percentile(word_lengths, 50)),
            "p95": int(np.percentile(word_lengths, 95)),
            "p99": int(np.percentile(word_lengths, 99)),
            "max": int(word_lengths.max()),
        },
    )

    if int(word_lengths.max()) > 140:
        raise RuntimeError(f"{split}: runaway raw text length. DO NOT TRAIN.")

    for d in DIMENSIONS:
        level_counts = Counter(
            r["labels"][d]["intensity_label"]
            for r in rows
            if r["labels"][d]["intensity_mask"] == 1
        )
        relative_only = sum(
            r["labels"][d]["presence"] == 1
            and r["labels"][d]["intensity_mask"] == 0
            for r in rows
        )
        distribution.append({
            "split": split,
            "dimension": d,
            "relative_only": relative_only,
            **{lv: level_counts.get(lv, 0) for lv in LEVELS},
        })

display(pd.DataFrame(distribution))
print("Distribution + raw length audit: PASS")


In [ ]:

# 7. LOAD TOKENIZER / BASE MODEL SOURCE
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

local_candidates = []
for config_path in KAGGLE_INPUT.rglob("config.json"):
    name = str(config_path.parent).casefold()
    if "xlm" in name and "roberta" in name:
        local_candidates.append(config_path.parent)

if local_candidates:
    BASE_MODEL_SOURCE = str(local_candidates[0])
    print("Using local XLM-R:", BASE_MODEL_SOURCE)
else:
    BASE_MODEL_SOURCE = BASE_MODEL_HF
    print("Using Hugging Face:", BASE_MODEL_SOURCE)
    print("If Internet is disabled, attach a local xlm-roberta-base Kaggle dataset.")

try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_SOURCE, use_fast=True)
except Exception as exc:
    raise RuntimeError(
        "Could not load XLM-R tokenizer. Enable Kaggle Internet or attach xlm-roberta-base."
    ) from exc

print("Tokenizer load: PASS")


In [ ]:

# 8. TOKEN LENGTH AUDIT — TRAIN + VALIDATION ONLY
def token_lengths(texts, batch_size=1024):
    lengths = []
    original_max = tokenizer.model_max_length
    tokenizer.model_max_length = 10_000_000

    try:
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start+batch_size]
            encoded = tokenizer(
                batch,
                add_special_tokens=True,
                truncation=False,
                padding=False,
                return_length=True,
                verbose=False,
            )
            lengths.extend(encoded["length"])
    finally:
        tokenizer.model_max_length = original_max

    return np.array(lengths, dtype=np.int32)

tv_texts = [r["text"] for r in train_rows] + [r["text"] for r in validation_rows]
lengths = token_lengths(tv_texts)

stats = {
    "p50": int(np.percentile(lengths, 50)),
    "p95": int(np.percentile(lengths, 95)),
    "p99": int(np.percentile(lengths, 99)),
    "p99_9": int(np.percentile(lengths, 99.9)),
    "max": int(lengths.max()),
}
print("Token length stats:", stats)

if stats["max"] > 512:
    raise RuntimeError("Unexpected sequence >512 XLM-R tokens. DO NOT TRAIN.")

MAX_LENGTH = None
for candidate in (128, 160, 192, 224, 256):
    truncation_rate = float(np.mean(lengths > candidate))
    print(f"candidate={candidate}: truncation={truncation_rate:.4%}")
    if truncation_rate <= 0.001:
        MAX_LENGTH = candidate
        break

if MAX_LENGTH is None:
    MAX_LENGTH = 256

final_truncation_rate = float(np.mean(lengths > MAX_LENGTH))
print("Selected MAX_LENGTH:", MAX_LENGTH)
print("Estimated TRAIN+VALIDATION truncation rate:", f"{final_truncation_rate:.4%}")

if final_truncation_rate > 0.001:
    raise RuntimeError("More than 0.1% truncation. DO NOT TRAIN.")

print("Token-length audit: PASS")


In [ ]:

# =========================
# 10. PRETOKENIZE TRAIN / VALIDATION / TEST
# =========================
from torch.utils.data import (
    Dataset,
    DataLoader,
)

def encode_rows(
    rows,
    batch_size=2048,
):
    input_ids_chunks = []
    attention_chunks = []

    texts = [
        row["text"]
        for row in rows
    ]

    for start in range(
        0,
        len(texts),
        batch_size,
    ):
        batch = texts[
            start:
            start + batch_size
        ]

        encoded = tokenizer(
            batch,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        input_ids_chunks.append(
            encoded["input_ids"]
        )
        attention_chunks.append(
            encoded["attention_mask"]
        )

    input_ids = torch.cat(
        input_ids_chunks,
        dim=0,
    )

    attention_mask = torch.cat(
        attention_chunks,
        dim=0,
    )

    presence = torch.tensor(
        [
            [
                row["labels"][dimension]["presence"]
                for dimension in DIMENSIONS
            ]
            for row in rows
        ],
        dtype=torch.float32,
    )

    intensity = torch.tensor(
        [
            [
                (
                    row["labels"][dimension]["intensity"]
                    if row["labels"][dimension]["intensity"] is not None
                    else -100
                )
                for dimension in DIMENSIONS
            ]
            for row in rows
        ],
        dtype=torch.long,
    )

    intensity_mask = torch.tensor(
        [
            [
                row["labels"][dimension]["intensity_mask"]
                for dimension in DIMENSIONS
            ]
            for row in rows
        ],
        dtype=torch.bool,
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "presence": presence,
        "intensity": intensity,
        "intensity_mask": intensity_mask,
    }

print("Encoding TRAIN...")
train_tensors = encode_rows(
    train_rows
)

print("Encoding VALIDATION...")
validation_tensors = encode_rows(
    validation_rows
)

print("Encoding TEST...")
test_tensors = encode_rows(
    test_rows
)

class TensorPreferenceDataset(
    Dataset
):
    def __init__(
        self,
        tensors,
    ):
        self.tensors = tensors

    def __len__(
        self,
    ):
        return self.tensors[
            "input_ids"
        ].shape[0]

    def __getitem__(
        self,
        index,
    ):
        return {
            key: value[index]
            for key, value
            in self.tensors.items()
        }

train_dataset = TensorPreferenceDataset(
    train_tensors
)
validation_dataset = TensorPreferenceDataset(
    validation_tensors
)
test_dataset = TensorPreferenceDataset(
    test_tensors
)

loader_generator = torch.Generator()
loader_generator.manual_seed(
    SEED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

print(
    "Encoded tensor shapes:",
    {
        "train": tuple(train_tensors["input_ids"].shape),
        "validation": tuple(validation_tensors["input_ids"].shape),
        "test": tuple(test_tensors["input_ids"].shape),
    },
)
print("Pretokenization: PASS")


In [ ]:

# =========================
# 11. XLM-R + 8 HEADS
# =========================
import torch.nn as nn
import torch.nn.functional as F

class XLMRPreferenceMultiTaskModel(
    nn.Module
):
    def __init__(
        self,
        encoder_source,
        dropout=0.10,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            encoder_source
        )

        hidden_size = int(
            self.encoder.config.hidden_size
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.presence_heads = nn.ModuleDict(
            {
                dimension: nn.Linear(
                    hidden_size,
                    1,
                )
                for dimension in DIMENSIONS
            }
        )

        # One ordinal head per dimension.
        # Four cumulative logits encode five ordered levels.
        self.intensity_heads = nn.ModuleDict(
            {
                dimension: nn.Linear(
                    hidden_size,
                    4,
                )
                for dimension in DIMENSIONS
            }
        )

    def forward(
        self,
        input_ids,
        attention_mask,
    ):
        output = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled = self.dropout(
            output.last_hidden_state[
                :,
                0,
                :,
            ]
        )

        presence_logits = torch.cat(
            [
                self.presence_heads[
                    dimension
                ](
                    pooled
                )
                for dimension in DIMENSIONS
            ],
            dim=1,
        )

        intensity_logits = torch.stack(
            [
                self.intensity_heads[
                    dimension
                ](
                    pooled
                )
                for dimension in DIMENSIONS
            ],
            dim=1,
        )

        return (
            presence_logits,
            intensity_logits,
        )

def unwrap_model(
    model,
):
    return (
        model.module
        if isinstance(
            model,
            nn.DataParallel,
        )
        else model
    )

print("Model architecture definition: PASS")


In [ ]:

# =========================
# 12. LOSS + METRICS
# =========================
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

train_presence_np = (
    train_tensors["presence"]
    .numpy()
)

positive_counts = (
    train_presence_np.sum(
        axis=0
    )
)

negative_counts = (
    train_presence_np.shape[0]
    - positive_counts
)

presence_pos_weight = torch.tensor(
    negative_counts
    / np.maximum(
        positive_counts,
        1.0,
    ),
    dtype=torch.float32,
    device=DEVICE,
)

print(
    "Presence pos_weight:",
    {
        dimension:
            float(
                presence_pos_weight[index].item()
            )
        for index, dimension
        in enumerate(DIMENSIONS)
    },
)

def ordinal_targets(
    intensity_labels,
):
    safe = intensity_labels.clamp(
        min=0,
        max=4,
    )

    thresholds = torch.arange(
        4,
        device=safe.device,
    )

    return (
        safe.unsqueeze(-1)
        > thresholds
    ).float()

def compute_loss(
    presence_logits,
    intensity_logits,
    presence_labels,
    intensity_labels,
    intensity_mask,
):
    presence_loss = (
        F.binary_cross_entropy_with_logits(
            presence_logits,
            presence_labels,
            pos_weight=presence_pos_weight,
        )
    )

    targets = ordinal_targets(
        intensity_labels
    )

    raw_intensity_loss = (
        F.binary_cross_entropy_with_logits(
            intensity_logits,
            targets,
            reduction="none",
        )
    ).mean(
        dim=-1
    )

    mask = intensity_mask.float()

    denominator = mask.sum().clamp(
        min=1.0
    )

    intensity_loss = (
        raw_intensity_loss
        * mask
    ).sum() / denominator

    cumulative = torch.sigmoid(
        intensity_logits
    )

    monotonic_violations = F.relu(
        cumulative[
            :,
            :,
            1:,
        ]
        - cumulative[
            :,
            :,
            :-1,
        ]
    ).mean(
        dim=-1
    )

    monotonicity_loss = (
        monotonic_violations
        * mask
    ).sum() / denominator

    total = (
        presence_loss
        + intensity_loss
        + MONOTONICITY_WEIGHT
        * monotonicity_loss
    )

    return {
        "total": total,
        "presence": presence_loss,
        "intensity": intensity_loss,
        "monotonicity": monotonicity_loss,
    }

def ordinal_probabilities_from_logits(
    logits,
):
    q = torch.sigmoid(
        logits
    )

    # Enforce cumulative monotonicity only for decoding.
    q = torch.cummin(
        q,
        dim=-1,
    ).values

    class_probabilities = torch.stack(
        [
            1.0 - q[..., 0],
            q[..., 0] - q[..., 1],
            q[..., 1] - q[..., 2],
            q[..., 2] - q[..., 3],
            q[..., 3],
        ],
        dim=-1,
    )

    class_probabilities = (
        class_probabilities.clamp(
            min=0.0
        )
    )

    class_probabilities = (
        class_probabilities
        / class_probabilities.sum(
            dim=-1,
            keepdim=True,
        ).clamp(
            min=1e-12
        )
    )

    return class_probabilities

def metrics_from_logits(
    presence_logits,
    intensity_logits,
    presence_gold,
    intensity_gold,
    intensity_mask,
):
    presence_probs = torch.sigmoid(
        presence_logits
    ).cpu().numpy()

    presence_pred = (
        presence_probs >= 0.5
    ).astype(
        np.int64
    )

    presence_gold_np = (
        presence_gold
        .cpu()
        .numpy()
        .astype(
            np.int64
        )
    )

    ordinal_probs = (
        ordinal_probabilities_from_logits(
            intensity_logits
        )
        .cpu()
        .numpy()
    )

    intensity_pred = (
        ordinal_probs.argmax(
            axis=-1
        )
    )

    intensity_gold_np = (
        intensity_gold
        .cpu()
        .numpy()
    )

    intensity_mask_np = (
        intensity_mask
        .cpu()
        .numpy()
        .astype(
            bool
        )
    )

    metrics = {
        "presence": {},
        "intensity": {},
    }

    per_dimension_presence_f1 = []

    for index, dimension in enumerate(
        DIMENSIONS
    ):
        y_true = presence_gold_np[
            :,
            index,
        ]

        y_pred = presence_pred[
            :,
            index,
        ]

        dim_metrics = {
            "accuracy":
                float(
                    accuracy_score(
                        y_true,
                        y_pred,
                    )
                ),
            "precision":
                float(
                    precision_score(
                        y_true,
                        y_pred,
                        zero_division=0,
                    )
                ),
            "recall":
                float(
                    recall_score(
                        y_true,
                        y_pred,
                        zero_division=0,
                    )
                ),
            "f1":
                float(
                    f1_score(
                        y_true,
                        y_pred,
                        zero_division=0,
                    )
                ),
        }

        metrics[
            "presence"
        ][dimension] = dim_metrics

        per_dimension_presence_f1.append(
            dim_metrics[
                "f1"
            ]
        )

    metrics[
        "presence_macro_f1"
    ] = float(
        np.mean(
            per_dimension_presence_f1
        )
    )

    all_true_presence = (
        presence_gold_np.reshape(-1)
    )

    all_pred_presence = (
        presence_pred.reshape(-1)
    )

    metrics[
        "presence_micro_f1"
    ] = float(
        f1_score(
            all_true_presence,
            all_pred_presence,
            zero_division=0,
        )
    )

    intensity_true_all = []
    intensity_pred_all = []

    per_dimension_intensity_f1 = []

    for index, dimension in enumerate(
        DIMENSIONS
    ):
        mask = intensity_mask_np[
            :,
            index,
        ]

        y_true = intensity_gold_np[
            mask,
            index,
        ]

        y_pred = intensity_pred[
            mask,
            index,
        ]

        if len(y_true) == 0:
            continue

        dim_metrics = {
            "accuracy":
                float(
                    accuracy_score(
                        y_true,
                        y_pred,
                    )
                ),
            "macro_f1":
                float(
                    f1_score(
                        y_true,
                        y_pred,
                        labels=list(
                            range(5)
                        ),
                        average="macro",
                        zero_division=0,
                    )
                ),
            "mae":
                float(
                    np.mean(
                        np.abs(
                            y_true
                            - y_pred
                        )
                    )
                ),
        }

        metrics[
            "intensity"
        ][dimension] = dim_metrics

        per_dimension_intensity_f1.append(
            dim_metrics[
                "macro_f1"
            ]
        )

        intensity_true_all.extend(
            y_true.tolist()
        )

        intensity_pred_all.extend(
            y_pred.tolist()
        )

    metrics[
        "intensity_macro_f1"
    ] = float(
        np.mean(
            per_dimension_intensity_f1
        )
    )

    metrics[
        "intensity_accuracy"
    ] = float(
        accuracy_score(
            intensity_true_all,
            intensity_pred_all,
        )
    )

    metrics[
        "intensity_mae"
    ] = float(
        np.mean(
            np.abs(
                np.array(
                    intensity_true_all
                )
                - np.array(
                    intensity_pred_all
                )
            )
        )
    )

    # All supervised information on a row must be correct:
    # - all four presence heads;
    # - all absolute intensities where intensity_mask=1.
    presence_row_correct = np.all(
        presence_pred
        == presence_gold_np,
        axis=1,
    )

    intensity_correct_matrix = (
        intensity_pred
        == intensity_gold_np
    )

    intensity_row_correct = np.all(
        np.logical_or(
            ~intensity_mask_np,
            intensity_correct_matrix,
        ),
        axis=1,
    )

    exact_supervised_state = np.mean(
        presence_row_correct
        & intensity_row_correct
    )

    metrics[
        "exact_supervised_state_match"
    ] = float(
        exact_supervised_state
    )

    normalized_mae_score = (
        1.0
        - min(
            1.0,
            metrics[
                "intensity_mae"
            ]
            / 4.0,
        )
    )

    metrics[
        "checkpoint_score"
    ] = float(
        0.40
        * metrics[
            "exact_supervised_state_match"
        ]
        + 0.30
        * metrics[
            "presence_macro_f1"
        ]
        + 0.20
        * metrics[
            "intensity_macro_f1"
        ]
        + 0.10
        * normalized_mae_score
    )

    return metrics

print("Loss + metrics definition: PASS")


In [ ]:

# =========================
# 13. TRAIN / EVALUATE FUNCTIONS
# =========================
from torch.optim import AdamW
from torch.amp import (
    autocast,
    GradScaler,
)
from tqdm.auto import tqdm

def collect_logits(
    model,
    loader,
):
    model.eval()

    presence_logits_all = []
    intensity_logits_all = []
    presence_gold_all = []
    intensity_gold_all = []
    intensity_mask_all = []

    total_loss = 0.0
    batches = 0

    with torch.no_grad():
        for batch in tqdm(
            loader,
            leave=False,
        ):
            input_ids = batch[
                "input_ids"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            presence = batch[
                "presence"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            intensity = batch[
                "intensity"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            intensity_mask = batch[
                "intensity_mask"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            with autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                (
                    presence_logits,
                    intensity_logits,
                ) = model(
                    input_ids,
                    attention_mask,
                )

                losses = compute_loss(
                    presence_logits,
                    intensity_logits,
                    presence,
                    intensity,
                    intensity_mask,
                )

            total_loss += float(
                losses["total"].item()
            )

            batches += 1

            presence_logits_all.append(
                presence_logits.detach().cpu()
            )

            intensity_logits_all.append(
                intensity_logits.detach().cpu()
            )

            presence_gold_all.append(
                presence.detach().cpu()
            )

            intensity_gold_all.append(
                intensity.detach().cpu()
            )

            intensity_mask_all.append(
                intensity_mask.detach().cpu()
            )

    return {
        "loss":
            total_loss
            / max(
                batches,
                1,
            ),
        "presence_logits":
            torch.cat(
                presence_logits_all,
                dim=0,
            ),
        "intensity_logits":
            torch.cat(
                intensity_logits_all,
                dim=0,
            ),
        "presence_gold":
            torch.cat(
                presence_gold_all,
                dim=0,
            ),
        "intensity_gold":
            torch.cat(
                intensity_gold_all,
                dim=0,
            ),
        "intensity_mask":
            torch.cat(
                intensity_mask_all,
                dim=0,
            ),
    }

def evaluate_model(
    model,
    loader,
):
    collected = collect_logits(
        model,
        loader,
    )

    metrics = metrics_from_logits(
        collected[
            "presence_logits"
        ],
        collected[
            "intensity_logits"
        ],
        collected[
            "presence_gold"
        ],
        collected[
            "intensity_gold"
        ],
        collected[
            "intensity_mask"
        ],
    )

    metrics[
        "loss"
    ] = collected[
        "loss"
    ]

    return (
        metrics,
        collected,
    )

def save_json(
    path,
    payload,
):
    Path(
        path
    ).write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )

print("Training utilities: PASS")


In [ ]:

# =========================
# 14. ONE-SHOT TRAINING
# =========================
if BEST_CHECKPOINT.exists() and not FORCE_RETRAIN:
    print(
        "Best checkpoint already exists."
    )
    print(
        "FORCE_RETRAIN=False => refusing to train a second time."
    )
    print(
        "Loading existing checkpoint instead."
    )

    raw_model = XLMRPreferenceMultiTaskModel(
        BASE_MODEL_SOURCE
    )

    state = torch.load(
        BEST_CHECKPOINT,
        map_location="cpu",
    )

    raw_model.load_state_dict(
        state
    )

else:
    raw_model = XLMRPreferenceMultiTaskModel(
        BASE_MODEL_SOURCE
    )

    if hasattr(
        raw_model.encoder,
        "gradient_checkpointing_enable",
    ):
        raw_model.encoder.gradient_checkpointing_enable()

    encoder_parameters = list(
        raw_model.encoder.parameters()
    )

    head_parameters = (
        list(
            raw_model.presence_heads.parameters()
        )
        + list(
            raw_model.intensity_heads.parameters()
        )
    )

    optimizer = AdamW(
        [
            {
                "params": encoder_parameters,
                "lr": ENCODER_LR,
                "weight_decay": WEIGHT_DECAY,
            },
            {
                "params": head_parameters,
                "lr": HEAD_LR,
                "weight_decay": WEIGHT_DECAY,
            },
        ]
    )

    updates_per_epoch = math.ceil(
        len(train_loader)
        / GRADIENT_ACCUMULATION
    )

    total_updates = (
        updates_per_epoch
        * MAX_EPOCHS
    )

    warmup_steps = int(
        total_updates
        * WARMUP_RATIO
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_updates,
        )
    )

    raw_model.to(
        DEVICE
    )

    model = nn.DataParallel(
        raw_model
    )

    scaler = GradScaler(
        "cuda"
    )

    best_score = -1.0
    epochs_without_improvement = 0
    history = []

    optimizer.zero_grad(
        set_to_none=True
    )

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        model.train()

        running = defaultdict(
            float
        )

        optimizer_steps = 0

        progress = tqdm(
            enumerate(
                train_loader,
                start=1,
            ),
            total=len(
                train_loader
            ),
            desc=f"Epoch {epoch}/{MAX_EPOCHS}",
        )

        for step, batch in progress:
            input_ids = batch[
                "input_ids"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            presence = batch[
                "presence"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            intensity = batch[
                "intensity"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            intensity_mask = batch[
                "intensity_mask"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            with autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                (
                    presence_logits,
                    intensity_logits,
                ) = model(
                    input_ids,
                    attention_mask,
                )

                losses = compute_loss(
                    presence_logits,
                    intensity_logits,
                    presence,
                    intensity,
                    intensity_mask,
                )

                loss_for_backward = (
                    losses["total"]
                    / GRADIENT_ACCUMULATION
                )

            scaler.scale(
                loss_for_backward
            ).backward()

            for key in (
                "total",
                "presence",
                "intensity",
                "monotonicity",
            ):
                running[key] += float(
                    losses[key].item()
                )

            should_step = (
                step
                % GRADIENT_ACCUMULATION
                == 0
                or step
                == len(train_loader)
            )

            if should_step:
                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    MAX_GRAD_NORM,
                )

                scaler.step(
                    optimizer
                )

                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

                scheduler.step()

                optimizer_steps += 1

            if step % 100 == 0:
                progress.set_postfix(
                    loss=(
                        running["total"]
                        / step
                    ),
                    lr=optimizer.param_groups[0]["lr"],
                )

        validation_metrics, _ = evaluate_model(
            model,
            validation_loader,
        )

        epoch_record = {
            "epoch": epoch,
            "train_loss":
                running["total"]
                / len(train_loader),
            "train_presence_loss":
                running["presence"]
                / len(train_loader),
            "train_intensity_loss":
                running["intensity"]
                / len(train_loader),
            "train_monotonicity_loss":
                running["monotonicity"]
                / len(train_loader),
            "validation":
                validation_metrics,
        }

        history.append(
            epoch_record
        )

        save_json(
            TRAINING_HISTORY_FILE,
            history,
        )

        score = validation_metrics[
            "checkpoint_score"
        ]

        print(
            f"\nEpoch {epoch} validation:"
        )

        print(
            json.dumps(
                validation_metrics,
                indent=2,
            )
        )

        if score > best_score + 1e-6:
            best_score = score
            epochs_without_improvement = 0

            cpu_state = {
                key:
                    value.detach().cpu()
                for key, value
                in unwrap_model(
                    model
                ).state_dict().items()
            }

            torch.save(
                cpu_state,
                BEST_CHECKPOINT,
            )

            print(
                "BEST CHECKPOINT SAVED:",
                BEST_CHECKPOINT,
            )

        else:
            epochs_without_improvement += 1

            print(
                "No improvement:",
                epochs_without_improvement,
                "/",
                EARLY_STOPPING_PATIENCE,
            )

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):
                print(
                    "Early stopping triggered."
                )
                break

    del model
    torch.cuda.empty_cache()

    # Reload the single best validation checkpoint.
    raw_model = XLMRPreferenceMultiTaskModel(
        BASE_MODEL_SOURCE
    )

    state = torch.load(
        BEST_CHECKPOINT,
        map_location="cpu",
    )

    raw_model.load_state_dict(
        state
    )

raw_model.to(
    DEVICE
)

model = nn.DataParallel(
    raw_model
)

best_validation_metrics, best_validation_collected = (
    evaluate_model(
        model,
        validation_loader,
    )
)

save_json(
    WORK_DIR
    / "best_validation_metrics.json",
    best_validation_metrics,
)

print(
    "\nBEST VALIDATION METRICS"
)
print(
    json.dumps(
        best_validation_metrics,
        indent=2,
    )
)


In [ ]:

# =========================
# 15. VALIDATION-ONLY SELECTIVE CALIBRATION
# =========================
presence_probs_val = (
    torch.sigmoid(
        best_validation_collected[
            "presence_logits"
        ]
    )
    .numpy()
)

presence_gold_val = (
    best_validation_collected[
        "presence_gold"
    ]
    .numpy()
    .astype(
        np.int64
    )
)

ordinal_probs_val = (
    ordinal_probabilities_from_logits(
        best_validation_collected[
            "intensity_logits"
        ]
    )
    .numpy()
)

intensity_pred_val = (
    ordinal_probs_val.argmax(
        axis=-1
    )
)

intensity_conf_val = (
    ordinal_probs_val.max(
        axis=-1
    )
)

intensity_gold_val = (
    best_validation_collected[
        "intensity_gold"
    ]
    .numpy()
)

intensity_mask_val = (
    best_validation_collected[
        "intensity_mask"
    ]
    .numpy()
    .astype(
        bool
    )
)

def calibrate_presence_band(
    probabilities,
    gold,
    target_accuracy,
):
    probabilities = np.asarray(
        probabilities
    )

    gold = np.asarray(
        gold
    )

    candidates = np.unique(
        np.concatenate(
            [
                np.linspace(
                    0.0,
                    0.50,
                    501,
                ),
                np.linspace(
                    0.50,
                    1.0,
                    501,
                ),
            ]
        )
    )

    best_negative = 0.0
    best_negative_coverage = 0

    for threshold in candidates[
        candidates <= 0.50
    ]:
        mask = probabilities <= threshold

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        accuracy = float(
            np.mean(
                gold[mask]
                == 0
            )
        )

        if (
            accuracy
            >= target_accuracy
            and count
            > best_negative_coverage
        ):
            best_negative = float(
                threshold
            )
            best_negative_coverage = count

    best_positive = 1.0
    best_positive_coverage = 0

    for threshold in candidates[
        candidates >= 0.50
    ]:
        mask = probabilities >= threshold

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        accuracy = float(
            np.mean(
                gold[mask]
                == 1
            )
        )

        if (
            accuracy
            >= target_accuracy
            and count
            > best_positive_coverage
        ):
            best_positive = float(
                threshold
            )
            best_positive_coverage = count

    if best_negative >= best_positive:
        best_negative = min(
            best_negative,
            0.25,
        )
        best_positive = max(
            best_positive,
            0.75,
        )

    direct_negative = (
        probabilities
        <= best_negative
    )

    direct_positive = (
        probabilities
        >= best_positive
    )

    direct = (
        direct_negative
        | direct_positive
    )

    direct_predictions = np.where(
        direct_positive,
        1,
        0,
    )

    direct_accuracy = (
        float(
            np.mean(
                direct_predictions[
                    direct
                ]
                == gold[
                    direct
                ]
            )
        )
        if direct.any()
        else None
    )

    return {
        "negative_max":
            best_negative,
        "positive_min":
            best_positive,
        "direct_coverage":
            float(
                direct.mean()
            ),
        "direct_accuracy":
            direct_accuracy,
        "fallback_rate":
            float(
                1.0
                - direct.mean()
            ),
    }

def calibrate_intensity_threshold(
    confidence,
    predictions,
    gold,
    mask,
    target_accuracy,
):
    confidence = np.asarray(
        confidence
    )[mask]

    predictions = np.asarray(
        predictions
    )[mask]

    gold = np.asarray(
        gold
    )[mask]

    if len(gold) == 0:
        return {
            "min_confidence": 1.0,
            "direct_coverage": 0.0,
            "direct_accuracy": None,
            "fallback_rate": 1.0,
        }

    candidates = np.linspace(
        0.0,
        1.0,
        1001,
    )

    best_threshold = 1.0
    best_coverage = 0

    for threshold in candidates:
        accepted = (
            confidence
            >= threshold
        )

        count = int(
            accepted.sum()
        )

        if count == 0:
            continue

        accuracy = float(
            np.mean(
                predictions[
                    accepted
                ]
                == gold[
                    accepted
                ]
            )
        )

        if (
            accuracy
            >= target_accuracy
            and count
            > best_coverage
        ):
            best_threshold = float(
                threshold
            )
            best_coverage = count

    accepted = (
        confidence
        >= best_threshold
    )

    direct_accuracy = (
        float(
            np.mean(
                predictions[
                    accepted
                ]
                == gold[
                    accepted
                ]
            )
        )
        if accepted.any()
        else None
    )

    return {
        "min_confidence":
            best_threshold,
        "direct_coverage":
            float(
                accepted.mean()
            ),
        "direct_accuracy":
            direct_accuracy,
        "fallback_rate":
            float(
                1.0
                - accepted.mean()
            ),
    }

calibration = {
    "status":
        "VALIDATION_ONLY_CALIBRATED_NOT_YET_TESTED",
    "target_direct_presence_accuracy":
        TARGET_DIRECT_PRESENCE_ACCURACY,
    "target_direct_intensity_accuracy":
        TARGET_DIRECT_INTENSITY_ACCURACY,
    "dimensions": {},
}

for index, dimension in enumerate(
    DIMENSIONS
):
    calibration[
        "dimensions"
    ][dimension] = {
        "presence":
            calibrate_presence_band(
                presence_probs_val[
                    :,
                    index,
                ],
                presence_gold_val[
                    :,
                    index,
                ],
                TARGET_DIRECT_PRESENCE_ACCURACY,
            ),
        "intensity":
            calibrate_intensity_threshold(
                intensity_conf_val[
                    :,
                    index,
                ],
                intensity_pred_val[
                    :,
                    index,
                ],
                intensity_gold_val[
                    :,
                    index,
                ],
                intensity_mask_val[
                    :,
                    index,
                ],
                TARGET_DIRECT_INTENSITY_ACCURACY,
            ),
    }

print(
    json.dumps(
        calibration,
        indent=2,
    )
)

save_json(
    WORK_DIR
    / "calibration_validation_only.json",
    calibration,
)

print("Validation-only calibration: PASS")


In [ ]:

# =========================
# 16. FREEZE MODEL + THRESHOLDS BEFORE TEST
# =========================
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Reload best CPU state and export compact fp16 safetensors.
best_state = torch.load(
    BEST_CHECKPOINT,
    map_location="cpu",
)

fp16_state = {}

for key, value in best_state.items():
    value = value.contiguous()

    if value.is_floating_point():
        value = value.half()

    fp16_state[
        key
    ] = value

model_file = (
    ARTIFACT_DIR
    / "model.safetensors"
)

save_safetensors(
    fp16_state,
    str(
        model_file
    ),
)

tokenizer_dir = (
    ARTIFACT_DIR
    / "tokenizer"
)

tokenizer.save_pretrained(
    tokenizer_dir
)

calibration[
    "status"
] = (
    "FROZEN_BEFORE_TEST"
)

calibration[
    "max_length"
] = MAX_LENGTH

calibration[
    "dimensions_order"
] = list(
    DIMENSIONS
)

save_json(
    ARTIFACT_DIR
    / "calibration.json",
    calibration,
)

architecture_config = {
    "architecture":
        "XLM-R Base shared encoder + 4 presence heads + 4 ordinal heads",
    "base_model":
        BASE_MODEL_HF,
    "dataset_version":
        EXPECTED_DATASET_VERSION,
    "dataset_zip_sha256":
        EXPECTED_DATASET_ZIP_SHA256,
    "dimensions":
        list(
            DIMENSIONS
        ),
    "absolute_levels":
        list(
            LEVELS
        ),
    "ordinal_encoding": {
        "VERY_LOW": [0, 0, 0, 0],
        "LOW": [1, 0, 0, 0],
        "MEDIUM": [1, 1, 0, 0],
        "HIGH": [1, 1, 1, 0],
        "VERY_HIGH": [1, 1, 1, 1],
    },
    "no_signal_rule":
        "NO_SIGNAL is presence=0, not an ordinal intensity class",
    "comparative_rule":
        "pure comparative dimensions have presence=1 and masked absolute intensity",
    "max_length":
        MAX_LENGTH,
}

save_json(
    ARTIFACT_DIR
    / "architecture_config.json",
    architecture_config,
)

shutil.copy2(
    WORK_DIR
    / "best_validation_metrics.json",
    ARTIFACT_DIR
    / "best_validation_metrics.json",
)

if TRAINING_HISTORY_FILE.exists():
    shutil.copy2(
        TRAINING_HISTORY_FILE,
        ARTIFACT_DIR
        / "training_history.json",
    )

freeze_manifest = {
    "status":
        "FROZEN_BEFORE_TEST",
    "model_sha256":
        sha256_file(
            model_file
        ),
    "best_checkpoint_sha256":
        sha256_file(
            BEST_CHECKPOINT
        ),
    "dataset_manifest_sha256":
        sha256_file(
            MANIFEST_FILE
        ),
    "dataset_qa_sha256":
        sha256_file(
            QA_FILE
        ),
    "max_length":
        MAX_LENGTH,
    "calibration":
        calibration,
    "test_seen_before_freeze":
        False,
    "final_holdout_seen_before_freeze":
        False,
}

save_json(
    FREEZE_FILE,
    freeze_manifest,
)

shutil.copy2(
    FREEZE_FILE,
    ARTIFACT_DIR
    / "freeze_manifest.json",
)

print(
    "MODEL + CALIBRATION FROZEN BEFORE TEST"
)
print(
    json.dumps(
        freeze_manifest,
        indent=2,
    )
)


In [ ]:

# =========================
# 17. TEST — ONE FROZEN RUN, NO TUNING AFTER THIS
# =========================
freeze_manifest = json.loads(
    FREEZE_FILE.read_text(
        encoding="utf-8"
    )
)

if (
    freeze_manifest[
        "status"
    ]
    != "FROZEN_BEFORE_TEST"
):
    raise RuntimeError(
        "Freeze status invalid. "
        "Do not evaluate TEST."
    )

test_metrics, test_collected = (
    evaluate_model(
        model,
        test_loader,
    )
)

def selective_metrics(
    collected,
    calibration,
):
    presence_probs = (
        torch.sigmoid(
            collected[
                "presence_logits"
            ]
        )
        .numpy()
    )

    presence_gold = (
        collected[
            "presence_gold"
        ]
        .numpy()
        .astype(
            np.int64
        )
    )

    ordinal_probs = (
        ordinal_probabilities_from_logits(
            collected[
                "intensity_logits"
            ]
        )
        .numpy()
    )

    intensity_pred = (
        ordinal_probs.argmax(
            axis=-1
        )
    )

    intensity_conf = (
        ordinal_probs.max(
            axis=-1
        )
    )

    intensity_gold = (
        collected[
            "intensity_gold"
        ]
        .numpy()
    )

    intensity_mask = (
        collected[
            "intensity_mask"
        ]
        .numpy()
        .astype(
            bool
        )
    )

    output = {
        "dimensions": {},
    }

    dimension_fallback_flags = np.zeros(
        presence_gold.shape,
        dtype=bool,
    )

    for index, dimension in enumerate(
        DIMENSIONS
    ):
        policy = calibration[
            "dimensions"
        ][dimension]

        negative_max = policy[
            "presence"
        ][
            "negative_max"
        ]

        positive_min = policy[
            "presence"
        ][
            "positive_min"
        ]

        intensity_min = policy[
            "intensity"
        ][
            "min_confidence"
        ]

        probs = presence_probs[
            :,
            index,
        ]

        gold_presence = presence_gold[
            :,
            index,
        ]

        direct_negative = (
            probs
            <= negative_max
        )

        direct_positive = (
            probs
            >= positive_min
        )

        direct_presence = (
            direct_negative
            | direct_positive
        )

        presence_prediction = np.where(
            direct_positive,
            1,
            0,
        )

        presence_direct_accuracy = (
            float(
                np.mean(
                    presence_prediction[
                        direct_presence
                    ]
                    == gold_presence[
                        direct_presence
                    ]
                )
            )
            if direct_presence.any()
            else None
        )

        fallback = (
            ~direct_presence
        )

        # A direct-positive dimension still falls back if its ordinal
        # intensity is not confident enough.
        gold_absolute = intensity_mask[
            :,
            index,
        ]

        needs_intensity = (
            direct_positive
            & gold_absolute
        )

        intensity_direct = (
            needs_intensity
            & (
                intensity_conf[
                    :,
                    index,
                ]
                >= intensity_min
            )
        )

        intensity_fallback = (
            needs_intensity
            & ~intensity_direct
        )

        fallback = (
            fallback
            | intensity_fallback
        )

        dimension_fallback_flags[
            :,
            index,
        ] = fallback

        intensity_direct_accuracy = (
            float(
                np.mean(
                    intensity_pred[
                        intensity_direct,
                        index,
                    ]
                    == intensity_gold[
                        intensity_direct,
                        index,
                    ]
                )
            )
            if intensity_direct.any()
            else None
        )

        output[
            "dimensions"
        ][dimension] = {
            "presence_direct_coverage":
                float(
                    direct_presence.mean()
                ),
            "presence_direct_accuracy":
                presence_direct_accuracy,
            "intensity_direct_coverage_on_absolute_cases":
                (
                    float(
                        intensity_direct.sum()
                        / max(
                            gold_absolute.sum(),
                            1,
                        )
                    )
                ),
            "intensity_direct_accuracy":
                intensity_direct_accuracy,
            "dimension_fallback_rate":
                float(
                    fallback.mean()
                ),
        }

    output[
        "dimension_fallback_rate_macro"
    ] = float(
        dimension_fallback_flags.mean()
    )

    output[
        "rows_requiring_any_llm_fallback"
    ] = float(
        dimension_fallback_flags.any(
            axis=1
        ).mean()
    )

    return output

test_selective = selective_metrics(
    test_collected,
    calibration,
)

test_report = {
    "status":
        "FROZEN_TEST_EVALUATED_NO_TUNING",
    "raw_transformer_metrics":
        test_metrics,
    "selective_transformer_policy":
        test_selective,
}

save_json(
    ARTIFACT_DIR
    / "test_metrics.json",
    test_report,
)

freeze_manifest[
    "status"
] = (
    "FROZEN_TEST_EVALUATED_NO_TUNING"
)

freeze_manifest[
    "test_seen_after_freeze"
] = True

save_json(
    FREEZE_FILE,
    freeze_manifest,
)

shutil.copy2(
    FREEZE_FILE,
    ARTIFACT_DIR
    / "freeze_manifest.json",
)

print(
    json.dumps(
        test_report,
        indent=2,
    )
)

print(
    "\nTEST evaluated after freeze."
)
print(
    "Do not change hyperparameters or thresholds based on TEST."
)


In [ ]:

# =========================
# 18. PACKAGE FINAL MODEL ZIP
# =========================
def rebuild_artifact_zip():
    if ARTIFACT_ZIP.exists():
        ARTIFACT_ZIP.unlink()

    with zipfile.ZipFile(
        ARTIFACT_ZIP,
        "w",
        zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as archive:
        for path in ARTIFACT_DIR.rglob(
            "*"
        ):
            if path.is_file():
                archive.write(
                    path,
                    path.relative_to(
                        ARTIFACT_DIR
                    ),
                )

    return sha256_file(
        ARTIFACT_ZIP
    )

artifact_zip_sha256 = (
    rebuild_artifact_zip()
)

print(
    "Artifact ZIP:",
    ARTIFACT_ZIP,
)
print(
    "Artifact ZIP SHA256:",
    artifact_zip_sha256,
)
print(
    "Artifact ZIP MB:",
    round(
        ARTIFACT_ZIP.stat().st_size
        / 1024
        / 1024,
        2,
    ),
)

from IPython.display import FileLink, display

display(
    FileLink(
        str(
            ARTIFACT_ZIP
        )
    )
)


In [ ]:

# =========================
# 19. OPTIONAL FINAL_HOLDOUT — ONE FINAL INDEPENDENT RUN
# =========================
if not RUN_FINAL_HOLDOUT:
    print(
        "FINAL_HOLDOUT NOT RUN."
    )
    print(
        "This is correct during training/calibration/test."
    )

else:
    freeze_manifest = json.loads(
        FREEZE_FILE.read_text(
            encoding="utf-8"
        )
    )

    if (
        freeze_manifest.get(
            "status"
        )
        != "FROZEN_TEST_EVALUATED_NO_TUNING"
    ):
        raise RuntimeError(
            "FINAL_HOLDOUT is allowed only after frozen TEST evaluation."
        )

    # This is the first notebook parse of FINAL_HOLDOUT.
    final_holdout_rows = read_jsonl(
        FILES[
            "final_holdout"
        ]
    )

    if (
        len(
            final_holdout_rows
        )
        != EXPECTED_SIZES[
            "final_holdout"
        ]
    ):
        raise RuntimeError(
            "FINAL_HOLDOUT size mismatch."
        )

    final_holdout_tensors = (
        encode_rows(
            final_holdout_rows
        )
    )

    final_holdout_dataset = (
        TensorPreferenceDataset(
            final_holdout_tensors
        )
    )

    final_holdout_loader = (
        DataLoader(
            final_holdout_dataset,
            batch_size=BATCH_SIZE * 2,
            shuffle=False,
            num_workers=2,
            pin_memory=True,
        )
    )

    final_metrics, final_collected = (
        evaluate_model(
            model,
            final_holdout_loader,
        )
    )

    final_selective = (
        selective_metrics(
            final_collected,
            calibration,
        )
    )

    final_report = {
        "status":
            "FINAL_HOLDOUT_FIRST_AND_ONLY_RUN",
        "raw_transformer_metrics":
            final_metrics,
        "selective_transformer_policy":
            final_selective,
    }

    save_json(
        ARTIFACT_DIR
        / "final_holdout_metrics.json",
        final_report,
    )

    freeze_manifest[
        "status"
    ] = (
        "FINAL_HOLDOUT_FIRST_AND_ONLY_RUN"
    )

    freeze_manifest[
        "final_holdout_seen_after_freeze"
    ] = True

    save_json(
        FREEZE_FILE,
        freeze_manifest,
    )

    shutil.copy2(
        FREEZE_FILE,
        ARTIFACT_DIR
        / "freeze_manifest.json",
    )

    final_zip_sha = (
        rebuild_artifact_zip()
    )

    print(
        json.dumps(
            final_report,
            indent=2,
        )
    )

    print(
        "\nFinal artifact ZIP SHA256:",
        final_zip_sha,
    )

    display(
        FileLink(
            str(
                ARTIFACT_ZIP
            )
        )
    )

    print(
        "\nFINAL_HOLDOUT complete."
    )
    print(
        "Do not retrain or recalibrate after this result."
    )



## Final holdout rule

Leave `RUN_FINAL_HOLDOUT=False` until the frozen TEST has completed.
Then set it to `True` and run only the final-holdout cell once.
Do not retrain after TEST or FINAL_HOLDOUT.
